# ingest-news-doe-pa-artran.ipynb

Diário Oficial do Estado do Pará — busca por termo (**ARTRAN**, Agência de
Regulação e Controle dos Serviços Públicos de Transporte do Estado do
Pará), via `ioepa.com.br/pesquisa`. Mesmo padrão do DOE-SP/SPI: busca
restrita ao dia de hoje; se não achar nada, é um dia normal sem
publicação (não é erro); se achar, baixa o Diário Completo (PDF) do dia.

Setor: Transporte.


In [0]:
%pip install --quiet httpx pypdf
dbutils.library.restartPython()

In [0]:
import os
import re
import json
import time
import random
import hashlib
import urllib.parse
from datetime import datetime
from io import BytesIO

import httpx
from pypdf import PdfReader


In [0]:
TERMO = "ARTRAN"
SEARCH_URL_BASE = "https://www.ioepa.com.br/pesquisa/"

SOURCE_ID = "doe_pa_artran"
SOURCE_DESCRICAO = "Diário Oficial do Estado do Pará — busca por termo (ARTRAN)"

_HOJE_DT = datetime.today()
HOJE_YYYYMMDD = _HOJE_DT.strftime("%Y%m%d")   # formato exigido pela URL de busca (di/df)
HOJE_ISO = _HOJE_DT.strftime("%Y-%m-%d")

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
HTTP_TIMEOUT = 60  # o PDF do diario completo pode ser grande

# Salva direto em files/{HOJE}, sem subpasta por setor (padrao corrigido).
PASTA_BASE = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE_ISO}"
PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_BASE, exist_ok=True)
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)

CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")


## Carrega a função compartilhada que atualiza o status real da fonte
na tabela de controle a cada execução.


In [0]:
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"

## Manifesto de deduplicação (chave: URL do PDF do Diário Completo)


In [0]:
def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    with open(caminho, "r", encoding="utf-8") as f:
        return set(json.load(f))


def salvar_manifesto(caminho: str, processados: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(processados), f, ensure_ascii=False, indent=2)


## Etapa 1 — Busca do termo, restrita ao dia de hoje (di=df=hoje)


In [0]:
def buscar_termo_hoje(termo: str, data_yyyymmdd: str) -> str:
    params = {"q": termo, "di": data_yyyymmdd, "df": data_yyyymmdd}
    resp = httpx.get(
        SEARCH_URL_BASE, params=params,
        headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.text


## Etapa 2 — Extrai os links de "Diário Completo" (PDF do dia inteiro)

O site linka duas coisas por resultado: a **página específica** onde o
termo aparece (`..._N.pdf`) e o **Diário Completo** (`..._0.pdf`, sempre
sufixo `_0`). Queremos só o segundo -- a data já vem embutida no próprio
nome do arquivo (`YYYY.MM.DD.DOE_0.pdf`), então não depende de conseguir
achar/parsear o texto "Diário publicado em: ..." ao redor do link.


In [0]:
PADRAO_PDF_COMPLETO = re.compile(r'href="([^"]+?(\d{4})\.(\d{2})\.(\d{2})\.DOE_0\.pdf)"')


def extrair_pdfs_completos(html: str) -> list[dict]:
    encontrados = {}
    for m in PADRAO_PDF_COMPLETO.finditer(html):
        url_relativa, ano, mes, dia = m.groups()
        url_absoluta = urllib.parse.urljoin("https://www.ioepa.com.br", url_relativa)
        data_iso = f"{ano}-{mes}-{dia}"
        encontrados[url_absoluta] = data_iso
    return [{"url": u, "date": d} for u, d in encontrados.items()]


## Etapa 3 — Baixa o PDF e extrai o texto (via pypdf)


In [0]:
def baixar_e_extrair_pdf(url: str) -> str:
    resp = httpx.get(
        url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, follow_redirects=True,
    )
    resp.raise_for_status()
    leitor = PdfReader(BytesIO(resp.content))
    paginas = [p.extract_text() or "" for p in leitor.pages]
    return "\n\n".join(paginas).strip()


## Salvamento — mesmo contrato de metadados usado em todo o projeto


In [0]:
def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    def _slugify(s: str, max_len: int = 60) -> str:
        import unicodedata
        s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
        s = re.sub(r"[^a-zA-Z0-9]+", "-", s).strip("-").lower()
        return (s[:max_len] or "sem-titulo").strip("-")

    nome_base = f"{SOURCE_ID}_{_slugify(titulo)}_{hashlib.md5(metadados['url'].encode()).hexdigest()[:8]}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


## Execução


In [0]:
try:
    ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)

    print(f"Buscando {TERMO!r} no Diário Oficial do PA, restrito a hoje ({HOJE_ISO})...")
    html = buscar_termo_hoje(TERMO, HOJE_YYYYMMDD)
    candidatos = extrair_pdfs_completos(html)

    # Defensivo: mesmo com o filtro di=df=hoje na URL de busca, só aceita
    # o que realmente bate com a data de hoje pelo nome do arquivo.
    candidatos_hoje = [c for c in candidatos if c["date"] == HOJE_ISO]
    print(f"{len(candidatos)} PDF(s) de \"Diário Completo\" encontrados; {len(candidatos_hoje)} de hoje.")

    if not candidatos_hoje:
        print(f"=== Fim. Nenhuma menção a {TERMO!r} publicada hoje. Dia normal, sem erro. ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=0)
    else:
        salvos = 0
        for item in candidatos_hoje:
            if item["url"] in ja_processados:
                print(f"  já processado antes, pulando: {item['url']}")
                continue

            print(f"  baixando Diário Completo de {item['date']}: {item['url']}")
            texto = baixar_e_extrair_pdf(item["url"])

            metadados = {
                "source_id": SOURCE_ID,
                "title": f"Diário Oficial do Pará — {item['date']} (menção a {TERMO})",
                "description": SOURCE_DESCRICAO,
                "url": item["url"],
                "date": item["date"],
                "published_at": item["date"],
            }
            salvar_artefatos(PASTA_BASE, metadados["title"], texto, metadados)
            ja_processados.add(item["url"])
            salvos += 1
            time.sleep(random.uniform(0.3, 0.8))

        salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)
        print(f"\n=== Fim. {salvos} novo(s) Diário(s) Completo(s) salvo(s) em {PASTA_BASE} ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=salvos)

except Exception as e:
    print(f"\n=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))
